# 🚀 Notebook 08 — Final Inference Pipeline

**Objective**: Build a single pipeline function that takes raw transaction data, preprocesses it, and outputs Fraud Risk Score + Typologies.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

MODEL_DIR = os.path.join('..', 'outputs', 'models')

# Load models
clf_is_aml = joblib.load(os.path.join(MODEL_DIR, 'lgbm_is_aml.pkl'))
clf_typology = joblib.load(os.path.join(MODEL_DIR, 'lgbm_typology.pkl'))
le_typology = joblib.load(os.path.join(MODEL_DIR, 'label_encoder_typology.pkl'))

# Model features
expected_features = clf_is_aml.feature_name_

def predict_transaction(transaction_df):
    # Preprocessing & Selection
    X = transaction_df.reindex(columns=expected_features).fillna(0)
    
    # 1. Predict AML Probability
    fraud_prob = clf_is_aml.predict_proba(X)[:, 1]
    
    # 2. Predict Typology Probabilities
    typology_probs = clf_typology.predict_proba(X)
    
    results = []
    for i in range(len(transaction_df)):
        txn_res = {
            'transaction_id': transaction_df.iloc[i].get('transaction_id', f'txn_{i}'),
            'fraud_risk_score': round(fraud_prob[i] * 100, 2),
            'risk_category': 'HIGH' if fraud_prob[i] > 0.8 else ('MEDIUM' if fraud_prob[i] > 0.4 else 'LOW'),
            'predicted_typology': le_typology.classes_[np.argmax(typology_probs[i])] if fraud_prob[i] > 0.5 else 'None'
        }
        
        if fraud_prob[i] > 0.5:
            # Top 3 typologies
            top_3_idx = np.argsort(typology_probs[i])[-3:][::-1]
            typologies = {le_typology.classes_[idx]: round(typology_probs[i][idx]*100, 2) for idx in top_3_idx}
            txn_res['typology_probabilities'] = typologies
        else:
            txn_res['typology_probabilities'] = {}
            
        results.append(txn_res)
        
    return results

print("Inference Pipeline Ready.")

Inference Pipeline Ready.


In [2]:
# Test the pipeline
DATA_DIR = os.path.join('..', 'data', 'raw')
sample_txn = pd.read_parquet(os.path.join(DATA_DIR, 'stg_transactions_features.parquet')).sample(5, random_state=42)

predictions = predict_transaction(sample_txn)
for p in predictions:
    print(p)

{'transaction_id': 'TXNGC4ZKZ4IGDSMQ8EC', 'fraud_risk_score': np.float64(2.99), 'risk_category': 'LOW', 'predicted_typology': 'None', 'typology_probabilities': {}}
{'transaction_id': 'TXNU2T9A5SJXQ430EVV', 'fraud_risk_score': np.float64(14.7), 'risk_category': 'LOW', 'predicted_typology': 'None', 'typology_probabilities': {}}
{'transaction_id': 'TXNRKP1G4HG4651PE95', 'fraud_risk_score': np.float64(2.73), 'risk_category': 'LOW', 'predicted_typology': 'None', 'typology_probabilities': {}}
{'transaction_id': 'TXNBJ0VQVGI1PAZTGI5', 'fraud_risk_score': np.float64(40.09), 'risk_category': 'MEDIUM', 'predicted_typology': 'None', 'typology_probabilities': {}}
{'transaction_id': 'TXNBDD58T3KRFRC1WXD', 'fraud_risk_score': np.float64(97.03), 'risk_category': 'HIGH', 'predicted_typology': 'Circular Transaction Loop', 'typology_probabilities': {'Circular Transaction Loop': np.float64(51.0), 'Rapid Multi-Hop Layering': np.float64(48.34), 'Underground Banking (Hawala)': np.float64(0.33)}}
